<a href="https://colab.research.google.com/github/Dharshini11042004/Adaptive-Threat-Detection-in-Smart-Healthcare-Infrastructure-with-Agentic-AI-using-RL/blob/main/Orchestrator_%26_Notification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Orchestrator + Email + Dashboard + Audit Logs
!pip -q install pandas numpy
import pandas as pd
import numpy as np
import smtplib
from email.mime.text import MIMEText
from datetime import datetime
from IPython.display import display, HTML
# LOAD AGENT OUTPUTS
a1 = pd.read_csv("/content/drive/MyDrive/MultiAgentModels/agent_1_outputs.csv")
a2 = pd.read_csv("/content/drive/MyDrive/MultiAgentModels/agent_2_outputs.csv")
a3 = pd.read_csv("/content/drive/MyDrive/MultiAgentModels/agent_3_outputs.csv")
a4 = pd.read_csv("/content/drive/MyDrive/MultiAgentModels/agent_4_outputs.csv")
min_len = min(len(a1), len(a2), len(a3), len(a4))
a1 = a1.iloc[:min_len]
a2 = a2.iloc[:min_len]
a3 = a3.iloc[:min_len]
a4 = a4.iloc[:min_len]
print("Aligned length:", min_len)
# EMAIL CONFIG
SENDER_EMAIL = "emaildharshini1104@gmail.com"
APP_PASSWORD = "bghtntmlfcjwfpfw"
RECEIVER_EMAIL = "emaildharshini1104@gmail.com"
def send_email(row):
    subject = f"🚨 [{row['Severity']}] Security Alert - {row['Decision']}"
    body = f"""
==============================
SECURITY OPERATIONS CENTER ALERT
===============================

A significant security event has been detected.

--------------------------------
THREAT SUMMARY
--------------------------------
Index: {row['Index']}
Severity: {row['Severity']}
Attack Type: {row['Attack Type']}
Final Risk Score: {round(row['Final Risk Score'],4)}

--------------------------------
COMPONENT RISK BREAKDOWN
--------------------------------
Network Risk: {round(row['Network Risk'],4)}
IoT Risk: {round(row['IoT Risk'],4)}
Host Risk: {round(row['Host Risk'],4)}
Healthcare Risk: {round(row['Healthcare Risk'],4)}

--------------------------------
SYSTEM DECISION
--------------------------------
Decision: {row['Decision']}
Action Taken: {row['Action']}
Autonomous Response: {row['Autonomous Response']}

--------------------------------
WHY THIS DECISION WAS MADE
--------------------------------
{row['Explanation']}

--------------------------------
RECOMMENDED ACTIONS
--------------------------------
• Investigate affected components immediately
• Review logs and abnormal patterns
• Apply manual override if required
• Escalate if repeated anomalies occur

--------------------------------
Generated by Multi-Agent AI SOC System
--------------------------------
"""
    try:
        msg = MIMEText(body)
        msg["Subject"] = subject
        msg["From"] = SENDER_EMAIL
        msg["To"] = RECEIVER_EMAIL
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(SENDER_EMAIL, APP_PASSWORD)
            server.sendmail(SENDER_EMAIL, RECEIVER_EMAIL, msg.as_string())
    except Exception as e:
        print("Email Error:", e)
# ADK DECISION ENGINE
def adk_agent(n, iot, h, hc):
    risk = (0.3*n + 0.25*iot + 0.25*h + 0.2*hc)
    if hc > 0.85:
        return "CRITICAL", risk, "patient safety is at immediate risk"
    if risk > 0.8:
        return "CRITICAL", risk, "multiple subsystems are severely compromised"
    elif risk > 0.6:
        return "RESTRICT", risk, "coordinated suspicious activity detected"
    elif risk > 0.3:
        return "MONITOR", risk, "moderate anomalies observed"
    else:
        return "ALLOW", risk, "system operating normally"
# SEVERITY CLASSIFICATION
def get_severity(risk):
    if risk > 0.8:
        return "CRITICAL"
    elif risk > 0.6:
        return "HIGH"
    elif risk > 0.3:
        return "MEDIUM"
    else:
        return "LOW"
# ATTACK TYPE CLASSIFICATION
def classify_attack(n, iot, h, hc):
    if hc > 0.7:
        return "Healthcare Data Breach"
    elif h > 0.7:
        return "Host Compromise"
    elif iot > 0.7:
        return "IoT Botnet Activity"
    elif n > 0.7:
        return "Network Intrusion"
    else:
        return "Normal / Low Risk Activity"
# EXPLANATION ENGINE
def explain(n, iot, h, hc, decision, reason):
    issues = []
    if n > 0.5:
        issues.append("network traffic")
    if iot > 0.5:
        issues.append("IoT device behaviour")
    if h > 0.5:
        issues.append("host system activity")
    if hc > 0.5:
        issues.append("patient data anomalies")
    if not issues:
        return "All monitored systems are behaving normally, so no action was required."
    return f"Issues were detected in {', '.join(issues)}. Since {reason}, the system decided to {decision.lower()} to prevent potential damage."
# ORCHESTRATION LOOP
records = []
audit_logs = []
critical_flag = False
health_flag = False
human_flag = False
for i in range(min_len):
    n = a1.iloc[i]["Risk Score"] / 100
    iot = a2.iloc[i]["Risk Score"] / 100
    h = a3.iloc[i]["Risk Score"] / 100
    hc = a4.iloc[i]["Risk Score"] / 100
    decision, risk, reason = adk_agent(n, iot, h, hc)
    severity = get_severity(risk)
    attack_type = classify_attack(n, iot, h, hc)
    if decision == "ALLOW":
        action = "No Action"
        response = "System Stable"
    elif decision == "MONITOR":
        action = "Observation"
        response = "Logging Enabled"
    elif decision == "RESTRICT":
        action = "Access Limited"
        response = "Partial Containment"
    else:
        action = "Abort"
        response = "Full Isolation"
    explanation = explain(n, iot, h, hc, decision, reason)
    # EMAIL TRIGGERS
    if hc > 0.85 and not health_flag:
        send_email({
            "Index": i, "Severity": severity, "Attack Type": attack_type,
            "Final Risk Score": risk,
            "Network Risk": n, "IoT Risk": iot,
            "Host Risk": h, "Healthcare Risk": hc,
            "Decision": decision, "Action": action,
            "Autonomous Response": response,
            "Explanation": explanation
        })
        health_flag = True
    if decision == "CRITICAL" and not critical_flag:
        send_email({
            "Index": i, "Severity": severity, "Attack Type": attack_type,
            "Final Risk Score": risk,
            "Network Risk": n, "IoT Risk": iot,
            "Host Risk": h, "Healthcare Risk": hc,
            "Decision": decision, "Action": action,
            "Autonomous Response": response,
            "Explanation": explanation
        })
        critical_flag = True
    if 0.55 < risk < 0.65 and not human_flag:
        send_email({
            "Index": i, "Severity": severity, "Attack Type": attack_type,
            "Final Risk Score": risk,
            "Network Risk": n, "IoT Risk": iot,
            "Host Risk": h, "Healthcare Risk": hc,
            "Decision": "HUMAN REQUIRED", "Action": "Review Needed",
            "Autonomous Response": "Pending",
            "Explanation": "The system is uncertain and requires human validation."
        })
        human_flag = True
    # STORE OUTPUT
    records.append({
        "Index": i,
        "Network Risk": n,
        "IoT Risk": iot,
        "Host Risk": h,
        "Healthcare Risk": hc,
        "Final Risk Score": risk,
        "Severity": severity,
        "Attack Type": attack_type,
        "Decision": decision,
        "Action": action,
        "Autonomous Response": response,
        "Explanation": explanation
    })
    audit_logs.append({
        "Time": str(datetime.now()),
        "Index": i,
        "Severity": severity,
        "Decision": decision,
        "Risk": risk
    })
# SAVE
df = pd.DataFrame(records)
df.to_csv("/content/drive/MyDrive/MultiAgentModels/final_orchestrator_output.csv", index=False)
pd.DataFrame(audit_logs).to_csv("/content/drive/MyDrive/MultiAgentModels/audit_logs.csv", index=False)
print("✅ Orchestrator Completed")

Aligned length: 21104
✅ Orchestrator Completed


In [ ]:
# ================================
# INSTALL DEPENDENCIES
# ================================
!pip -q install streamlit pyngrok pandas plotly numpy

# ================================
# CREATE DASHBOARD FILE
# ================================
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

st.set_page_config(page_title="Healthcare SOC", layout="wide")

st.title("🏥 SMART HEALTHCARE SECURITY OPERATIONS CENTER")

# LOAD DATA
df = pd.read_csv("/content/drive/MyDrive/MultiAgentModels/final_orchestrator_output.csv")

# ================================
# COLOR MAPPING
# ================================
severity_colors = {
    "LOW": "green",
    "MEDIUM": "yellow",
    "HIGH": "orange",
    "CRITICAL": "red"
}

# ================================
# KPI METRICS
# ================================
total = len(df)
low = len(df[df["Severity"]=="LOW"])
medium = len(df[df["Severity"]=="MEDIUM"])
high = len(df[df["Severity"]=="HIGH"])
critical = len(df[df["Severity"]=="CRITICAL"])

col1, col2, col3, col4, col5 = st.columns(5)
col1.metric("Total Events", total)
col2.metric("Low", low)
col3.metric("Medium", medium)
col4.metric("High", high)
col5.metric("Critical", critical)

st.markdown("---")

# ================================
# SEVERITY DISTRIBUTION
# ================================
st.subheader("📊 Threat Severity Distribution")

fig1 = px.pie(
    df,
    names="Severity",
    color="Severity",
    color_discrete_map=severity_colors
)
st.plotly_chart(fig1, use_container_width=True)

# ================================
# ATTACK TYPE DISTRIBUTION
# ================================
st.subheader("⚠️ Attack Type Distribution")

attack_counts = df["Attack Type"].value_counts().reset_index()
attack_counts.columns = ["Attack Type", "Count"]

fig2 = px.bar(
    attack_counts,
    x="Attack Type",
    y="Count",
    color="Attack Type"
)
st.plotly_chart(fig2, use_container_width=True)

# ================================
# BASIC RISK TREND
# ================================
st.subheader("📈 Risk Score Trend")

fig3 = px.line(
    df.head(500),
    y="Final Risk Score"
)
st.plotly_chart(fig3, use_container_width=True)

# ================================
# 🔬 STOCHASTIC ANALYSIS
# ================================
st.markdown("## 🔬 Stochastic Risk Analysis")

window = st.slider("Rolling Window Size", 5, 50, 20)

df["Rolling Mean"] = df["Final Risk Score"].rolling(window).mean()
df["Rolling Std"] = df["Final Risk Score"].rolling(window).std()

# Z-score
df["Z-Score"] = (df["Final Risk Score"] - df["Final Risk Score"].mean()) / df["Final Risk Score"].std()

subset = df.head(500)

# ================================
# CONFIDENCE BAND
# ================================
st.subheader("📉 Risk with Confidence Band")

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    y=subset["Rolling Mean"],
    mode='lines',
    name='Rolling Mean'
))

fig4.add_trace(go.Scatter(
    y=subset["Rolling Mean"] + subset["Rolling Std"],
    mode='lines',
    name='Upper Bound',
    line=dict(dash='dash')
))

fig4.add_trace(go.Scatter(
    y=subset["Rolling Mean"] - subset["Rolling Std"],
    mode='lines',
    name='Lower Bound',
    line=dict(dash='dash')
))

st.plotly_chart(fig4, use_container_width=True)

# ================================
# Z-SCORE ANOMALY DETECTION
# ================================
st.subheader("🚨 Statistical Anomalies")

anomalies = subset[abs(subset["Z-Score"]) > 3]

fig5 = px.scatter(
    subset,
    y="Final Risk Score"
)

fig5.add_scatter(
    y=anomalies["Final Risk Score"],
    mode='markers',
    name='Anomalies'
)

st.plotly_chart(fig5, use_container_width=True)
st.write(f"Detected Anomalies: {len(anomalies)}")

# ================================
# ENTROPY
# ================================
st.subheader("🧠 System Uncertainty (Entropy)")

prob = df["Severity"].value_counts(normalize=True)
entropy = -np.sum(prob * np.log2(prob + 1e-9))

st.metric("Entropy", round(entropy, 3))

# ================================
# CRITICAL ALERT PANEL
# ================================
st.subheader("🚨 Critical Alerts")

critical_df = df[df["Severity"]=="CRITICAL"]

if len(critical_df) > 0:
    for i, row in critical_df.head(5).iterrows():
        st.error(f"""
🔴 INDEX: {row['Index']}

Attack Type: {row['Attack Type']}
Risk Score: {round(row['Final Risk Score'],3)}
Decision: {row['Decision']}

Reason:
{row['Explanation']}
""")
else:
    st.success("No critical alerts detected")

# ================================
# HUMAN-IN-THE-LOOP
# ================================
st.subheader("🧠 Human Intervention Panel")

idx = st.number_input("Select Event Index", 0, len(df)-1)

current = df.loc[idx]

st.info(f"""
Current Decision: {current['Decision']}
Severity: {current['Severity']}
Attack Type: {current['Attack Type']}
""")

new_decision = st.selectbox(
    "Override Decision",
    ["ALLOW","MONITOR","RESTRICT","CRITICAL"]
)

if st.button("Apply Override"):
    df.loc[idx, "Decision"] = new_decision
    df.loc[idx, "Autonomous Response"] = "Human Override"
    df.loc[idx, "Explanation"] = "Decision overridden by human analyst"

    df.to_csv("/content/drive/MyDrive/MultiAgentModels/final_orchestrator_output.csv", index=False)

    st.success("✅ Override applied successfully")
'''

with open("dashboard.py", "w") as f:
    f.write(dashboard_code)

# ================================
# START NGROK
# ================================
from pyngrok import ngrok

# ⚠️ Replace with your own token if needed
ngrok.set_auth_token("3BLDNEpT5Fnxnc8cnEOJaTy1oeo_5audz528vtLjv2viuFged")

ngrok.kill()

public_url = ngrok.connect(8501)
print("🚀 Dashboard Link:", public_url)

# ================================
# RUN STREAMLIT
# ================================
!streamlit run dashboard.py &>/dev/null &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 79.4 MB/s eta 0:00:00
🚀 Dashboard Link: NgrokTunnel: "https://nonobservational-ariana-sliest.ngrok-free.dev" -> "http://localhost:8501"
